# Router Adapter Inference

This notebook keeps only the thin inference path: bootstrap, optional dependency install, image upload, and `RouterAdapterRuntime` prediction.

The returned payload keeps the mirrored top-level crop fields and also includes a structured `router` block with router status, primary detection, and detection count.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def refresh_colab_clone(repo_root: Path) -> None:
    repo_root = Path(repo_root)
    if repo_root != CLONE_TARGET or not (repo_root / '.git').exists():
        return
    try:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', REPO_URL], check=False)
        subprocess.run(['git', '-C', str(repo_root), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'checkout', 'master'], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only', 'origin', 'master'], check=True)
        print(f'Refreshed Colab clone at {repo_root}')
    except Exception as exc:
        print(f'Warning: could not refresh Colab clone at {repo_root}: {exc}')

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab
except ModuleNotFoundError:
    if not (CLONE_TARGET / 'scripts').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    os.chdir(CLONE_TARGET)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    refresh_colab_clone(ROOT)
    mount_drive_if_available()
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)

In [ ]:
from scripts.colab_repo_bootstrap import login_and_check_hf_token, running_in_colab
from scripts.colab_router_adapter_inference import run_inference

# Notebook-first inference controls (no codebase edits needed).
NOTEBOOK_PROFILE = 'a100_colab_default'  # one of: custom, a100_colab_default, cpu_debug

INFERENCE_PROFILES = {
    'custom': {},
    'a100_colab_default': {
        'CONFIG_ENV': 'colab',
        'DEVICE': 'cuda',
        'REQUIRE_HF_LOGIN': False,
        'CROP_HINT': None,
        'PART_HINT': None,
        'ADAPTER_ROOT': None,
        'FORCE_UPLOAD_IF_NO_IMAGE': True,
    },
    'cpu_debug': {
        'CONFIG_ENV': 'colab',
        'DEVICE': 'cpu',
        'REQUIRE_HF_LOGIN': False,
        'CROP_HINT': None,
        'PART_HINT': None,
        'ADAPTER_ROOT': None,
        'FORCE_UPLOAD_IF_NO_IMAGE': False,
    },
}

CONFIG_ENV = 'colab'
DEVICE = 'cuda'
REQUIRE_HF_LOGIN = False
CROP_HINT = None
PART_HINT = None
ADAPTER_ROOT = None
IMAGE_PATH = None  # Set directly to skip upload.
FORCE_UPLOAD_IF_NO_IMAGE = True

profile = dict(INFERENCE_PROFILES.get(NOTEBOOK_PROFILE, {}))
CONFIG_ENV = str(profile.get('CONFIG_ENV', CONFIG_ENV))
DEVICE = str(profile.get('DEVICE', DEVICE))
REQUIRE_HF_LOGIN = bool(profile.get('REQUIRE_HF_LOGIN', REQUIRE_HF_LOGIN))
CROP_HINT = profile.get('CROP_HINT', CROP_HINT)
PART_HINT = profile.get('PART_HINT', PART_HINT)
ADAPTER_ROOT = profile.get('ADAPTER_ROOT', ADAPTER_ROOT)
FORCE_UPLOAD_IF_NO_IMAGE = bool(profile.get('FORCE_UPLOAD_IF_NO_IMAGE', FORCE_UPLOAD_IF_NO_IMAGE))

# Optional final overrides with highest precedence.
INFERENCE_OVERRIDES = {}
if INFERENCE_OVERRIDES:
    CONFIG_ENV = str(INFERENCE_OVERRIDES.get('CONFIG_ENV', CONFIG_ENV))
    DEVICE = str(INFERENCE_OVERRIDES.get('DEVICE', DEVICE))
    REQUIRE_HF_LOGIN = bool(INFERENCE_OVERRIDES.get('REQUIRE_HF_LOGIN', REQUIRE_HF_LOGIN))
    CROP_HINT = INFERENCE_OVERRIDES.get('CROP_HINT', CROP_HINT)
    PART_HINT = INFERENCE_OVERRIDES.get('PART_HINT', PART_HINT)
    ADAPTER_ROOT = INFERENCE_OVERRIDES.get('ADAPTER_ROOT', ADAPTER_ROOT)
    IMAGE_PATH = INFERENCE_OVERRIDES.get('IMAGE_PATH', IMAGE_PATH)
    FORCE_UPLOAD_IF_NO_IMAGE = bool(INFERENCE_OVERRIDES.get('FORCE_UPLOAD_IF_NO_IMAGE', FORCE_UPLOAD_IF_NO_IMAGE))

print(
    f'[CONFIG] profile={NOTEBOOK_PROFILE} env={CONFIG_ENV} device={DEVICE} '
    f'crop_hint={CROP_HINT} part_hint={PART_HINT}'
)

HF_READY = login_and_check_hf_token(print_fn=print)
if REQUIRE_HF_LOGIN and not HF_READY:
    raise RuntimeError('HF login required by notebook settings, but authentication failed.')
if not HF_READY:
    print('[HF] Continuing without an authenticated session. Gated model downloads may fail.')

if IMAGE_PATH is None and FORCE_UPLOAD_IF_NO_IMAGE:
    if not running_in_colab():
        raise ValueError('Set IMAGE_PATH manually when running outside Colab.')
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded.keys()))

if IMAGE_PATH is None:
    raise ValueError('IMAGE_PATH is required when upload is disabled.')

# The returned payload includes `router.status`, `router.primary_detection`, and mirrored top-level crop fields.
result = run_inference(
    IMAGE_PATH,
    config_env=CONFIG_ENV,
    crop_hint=CROP_HINT,
    part_hint=PART_HINT,
    adapter_root=ADAPTER_ROOT,
    device=DEVICE,
    status_printer=print,
)
result